In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (ConfusionMatrixDisplay, auc, confusion_matrix, roc_curve, silhouette_samples)
from sklearn.model_selection import cross_val_predict
from sklearn.preprocessing import label_binarize

import kmeans_communes as km

sns.set_theme(style="whitegrid", context="talk", palette="viridis")
OUT = km.OUT / "graphiques"
OUT.mkdir(parents=True, exist_ok=True)

def sauver(nom):
    plt.tight_layout()
    plt.savefig(OUT / f"{nom}.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("OK ->", nom)

# =============== 1. RÉPARTITION : histogramme + boîte à moustaches ===============
def g_histogramme(df):
    p = df["prix_m2_fiable"].clip(upper=df["prix_m2_fiable"].quantile(0.995))
    plt.figure(figsize=(11, 6))
    sns.histplot(p, bins=80, color="#2a788e")
    plt.axvline(p.median(), color="red", ls="--", label=f"médiane = {p.median():,.0f} €/m²")
    plt.xlabel("Prix au m² (€)"); plt.ylabel("Nombre de mutations")
    plt.title("Répartition du prix au m² (France)"); plt.legend()
    sauver("01_histogramme_prix_m2")

def g_boxplot_region(df):
    ordre = df.groupby("region")["prix_m2_fiable"].median().sort_values(ascending=False).index
    plt.figure(figsize=(13, 7))
    sns.boxplot(data=df, y="region", x="prix_m2_fiable", order=ordre, showfliers=False, color="#5ec962")
    plt.xlabel("Prix au m² (€)"); plt.ylabel("")
    plt.title("Dispersion du prix au m² par région")
    sauver("02_boxplot_prix_par_region")

# =============== 2. COMPARAISON : barres ===============
def g_barres_region(df):
    r = df.groupby("region").agg(prix=("prix_m2_fiable", "median"), volume=("prix_m2_fiable", "size")).sort_values("prix")
    fig, ax = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
    ax[0].barh(r.index, r["prix"], color="#2a788e"); ax[0].set_title("Prix médian (€/m²)")
    ax[1].barh(r.index, r["volume"], color="#7ad151"); ax[1].set_title("Nombre de transactions (liquidité)")
    sauver("03_barres_region_prix_volume")

def g_barres_dept(df):
    d = (df.groupby("code_dept")["prix_m2_fiable"].agg(["median", "size"]).query("size >= 200").sort_values("median", ascending=False).head(15))
    plt.figure(figsize=(11, 7))
    sns.barplot(x=d["median"], y=d.index, color="#2a788e")
    plt.xlabel("Prix médian (€/m²)"); plt.ylabel("Département")
    plt.title("Top 15 départements les plus chers")
    sauver("04_barres_top_departements")

# =============== 3. ÉVOLUTION : courbe + aire (par mois sur 2025) ===============
def g_evolution(df):
    m = df.groupby("annee_mois").agg(prix=("prix_m2_fiable", "median"), volume=("prix_m2_fiable", "size")).reset_index()
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.fill_between(m["annee_mois"], m["volume"], color="#7ad151", alpha=0.4)
    ax.set_ylabel("Transactions / mois")
    ax2 = ax.twinx()
    ax2.plot(m["annee_mois"], m["prix"], color="#440154", marker="o", lw=3)
    ax2.set_ylabel("Prix médian (€/m²)"); ax2.grid(False)
    ax.tick_params(axis="x", rotation=45)
    plt.title("Évolution mensuelle : volume (aire) et prix (courbe)")
    sauver("05_evolution_mensuelle")

# =============== 4. CORRÉLATION : nuages de points + heatmap ===============
def g_scatter(c):
    fig, ax = plt.subplots(1, 2, figsize=(17, 7))
    sns.scatterplot(data=c, x="prix_m2_med", y="ticket_med", hue="cluster", palette="viridis", s=12, alpha=0.6, ax=ax[0])
    ax[0].set_xscale("log"); ax[0].set_yscale("log")
    ax[0].set_title("Prix/m² vs ticket médian")
    sns.scatterplot(data=c, x="prix_m2_med", y="n_tx", hue="cluster", palette="viridis", s=12, alpha=0.6, ax=ax[1], legend=False)
    ax[1].set_xscale("log"); ax[1].set_yscale("log")
    ax[1].set_title("Prix/m² vs volume (liquidité)")
    sauver("06_nuages_de_points")

def g_correlation(c):
    cols = ["prix_m2_med", "ticket_med", "n_tx", "surf_med", "dispersion"]
    plt.figure(figsize=(8, 7))
    sns.heatmap(c[cols].corr(method="spearman"), annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1)
    plt.title("Corrélations (Spearman) entre variables")
    sauver("07_matrice_correlation")

# =============== 5. QUALITÉ DU CLUSTERING (« courbe de perte ») ===============
def g_choix_k():
    m = pd.read_csv(km.OUT / "metriques_k.csv")
    k_ret = int(m.loc[m["k_retenu"], "k"].iloc[0])
    fig, ax = plt.subplots(2, 3, figsize=(20, 11))
    ax = ax.ravel()
    ax[0].plot(m["k"], m["inertie"], marker="o"); ax[0].set_title("Inertie (méthode du coude)")
    ax[1].plot(m["k"], m["silhouette"], marker="o", color="green"); ax[1].set_title("Silhouette (↑)")
    ax[2].plot(m["k"], m["davies_bouldin"], marker="o", color="red"); ax[2].set_title("Davies-Bouldin (↓)")
    ax[3].plot(m["k"], m["calinski_harabasz"], marker="o", color="purple"); ax[3].set_title("Calinski-Harabasz (↑)")
    ax[4].errorbar(m["k"], m["stabilite_ari"], yerr=m["stabilite_std"], marker="o", color="orange", capsize=4)
    ax[4].axhline(km.ARI_MIN, ls=":", color="grey"); ax[4].set_title("Stabilité bootstrap (ARI, ↑)")
    ax[5].plot(m["k"], m["score_global"], marker="o", color="black"); ax[5].set_title("Score global (règle de choix)")
    for a in ax:
        a.set_xlabel("k"); a.axvline(k_ret, color="red", ls="--", alpha=0.5)
    plt.suptitle(f"Choix de k : k retenu = {k_ret}", fontsize=22)
    sauver("08_choix_de_k")

def g_silhouette(X, labels):
    idx = np.random.RandomState(km.SEED).choice(len(X), min(5000, len(X)), replace=False)
    s = silhouette_samples(X[idx], labels[idx]); l = labels[idx]
    plt.figure(figsize=(10, 7)); y = 0
    for k in np.unique(l):
        v = np.sort(s[l == k])
        plt.fill_betweenx(np.arange(y, y + len(v)), 0, v, alpha=0.7, label=f"Cluster {k}")
        y += len(v) + 50
    plt.axvline(s.mean(), color="red", ls="--", label=f"moyenne = {s.mean():.2f}")
    plt.xlabel("Coefficient de silhouette"); plt.yticks([]); plt.legend()
    plt.title("Silhouette par cluster")
    sauver("09_silhouette_par_cluster")

def g_pca(X, labels):
    Z = PCA(n_components=2, random_state=km.SEED).fit_transform(X)
    plt.figure(figsize=(10, 8))
    sns.scatterplot(x=Z[:, 0], y=Z[:, 1], hue=labels, palette="viridis", s=10, alpha=0.6)
    plt.xlabel("Composante 1"); plt.ylabel("Composante 2")
    plt.title("Clusters projetés en 2D (ACP)")
    sauver("10_acp_clusters")

# =============== 6. PROFIL DES SEGMENTS ===============
def g_profil(c):
    cols = ["prix_m2_med", "ticket_med", "n_tx", "surf_med", "dispersion"]
    z = c.groupby("cluster")[cols].median()
    z = (z - z.mean()) / z.std()
    plt.figure(figsize=(10, 6))
    sns.heatmap(z, annot=True, fmt=".1f", cmap="RdYlGn", center=0)
    plt.title("Profil des clusters (écart à la moyenne, en σ)")
    sauver("11_profil_clusters")

    plt.figure(figsize=(9, 5))
    c["cluster"].value_counts().sort_index().plot.bar(color="#2a788e")
    plt.xlabel("Cluster"); plt.ylabel("Nb de communes"); plt.xticks(rotation=0)
    plt.title("Taille des clusters")
    sauver("12_taille_clusters")

# =============== 7. CONFUSION + ROC (validité des clusters) ===============
def g_confusion_roc(X, labels):
    """k-means est non supervisé : on mesure la séparabilité des clusters avec
    un classifieur prédisant le cluster par validation croisée."""
    rf = RandomForestClassifier(n_estimators=150, random_state=km.SEED, n_jobs=-1)
    proba = cross_val_predict(rf, X, labels, cv=5, method="predict_proba")
    pred = proba.argmax(axis=1)
    classes = np.unique(labels)

    cm = confusion_matrix(labels, pred, normalize="true")
    fig, ax = plt.subplots(figsize=(8, 7))
    ConfusionMatrixDisplay(cm, display_labels=classes).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
    plt.title("Matrice de confusion (clusters, validation croisée)")
    sauver("13_matrice_confusion")

    Y = label_binarize(labels, classes=classes)
    plt.figure(figsize=(9, 7))
    for i, k in enumerate(classes):
        fpr, tpr, _ = roc_curve(Y[:, i], proba[:, i])
        plt.plot(fpr, tpr, label=f"Cluster {k} (AUC = {auc(fpr, tpr):.3f})")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("Taux de faux positifs"); plt.ylabel("Taux de vrais positifs")
    plt.title("Courbes ROC (un cluster contre les autres)"); plt.legend()
    sauver("14_courbes_roc")

def main():
    df = km.charger_mutations()
    c = pd.read_csv(km.OUT / "communes_clusters.csv", dtype={"code_insee": str, "code_dept": str})
    X, _ = km.construire_X(c)
    labels = c["cluster"].to_numpy()

    g_histogramme(df); g_boxplot_region(df)
    g_barres_region(df); g_barres_dept(df)
    g_evolution(df)
    g_scatter(c); g_correlation(c)
    g_choix_k(); g_silhouette(X, labels); g_pca(X, labels)
    g_profil(c)
    g_confusion_roc(X, labels)
    print("\nTerminé ->", OUT.resolve())

if __name__ == "__main__":
    main()

OK -> 01_histogramme_prix_m2
OK -> 02_boxplot_prix_par_region
OK -> 03_barres_region_prix_volume
OK -> 04_barres_top_departements
OK -> 05_evolution_mensuelle
OK -> 06_nuages_de_points
OK -> 07_matrice_correlation
OK -> 08_choix_de_k
OK -> 09_silhouette_par_cluster
OK -> 10_acp_clusters
OK -> 11_profil_clusters
OK -> 12_taille_clusters
OK -> 13_matrice_confusion
OK -> 14_courbes_roc

Terminé -> C:\Users\pitsy\Desktop\premier cours année 5 python\Data-Visualisation-Projet\sortie\graphiques
